# Greenville, South Carolina — City Levy, 4:1 Split-Rate LVT

This notebook models a revenue-neutral shift of the **City of Greenville's own property-tax levy**
(81.4 mills) from the current uniform tax on land + buildings to a **4:1 split-rate**, where land
is taxed at 4× the rate on buildings. County, school-district, and special-district levies stay as
they are and are not modeled.

**Scope decisions (agreed up front)**

| Decision | Choice |
|---|---|
| Geography | Parcels whose centroid falls inside the City of Greenville boundary polygon (Census TIGERweb, current vintage) |
| Levy | City of Greenville levy only: 81.4 mills (Greenville County Auditor millage sheet, tax year 2021) |
| Reform | 4:1 split-rate, revenue-neutral on the modeled taxable base |
| SC assessment ratios | Kept: 4% for owner-occupied legal residences, 6% for all other locally assessed real property. The ratio applies to **both** land and buildings, so only the millage is split |
| Existing relief | Kept: legal-residence 4% ratio, the 65+/disabled/blind homestead exemption, full exemptions (government, church, charity, hospital, etc.), and the Act 388 15% reassessment cap (the county's *taxable* market value is used, not fair market value) |
| Homestead | Modeled at **$75,000** (vs. today's $50,000) with today's eligibility (65+, totally disabled, legally blind; legal residence). The baseline and the LVT scenario both use $75K; Section 6 also reports the $50K→$75K change under today's system |
| FILOT / special arrangements | Held out of the shift; TIF parcels are modeled like any other parcel |

## Section 1 — Imports, constants, `.env`

In [1]:
import sys
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

# Constants
CITY_NAME = 'greenville_sc'
STATE_FIPS = '45'
COUNTY_FIPS = '045'
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

TAX_YEAR = 2021                 # vintage of the county parcel snapshot (values + bills)
CITY_MILLAGE = 81.4             # City of Greenville levy, Greenville County 2021 millage sheet (district 500)
DISTRICT_500_TOTAL_MILLAGE = 339.3  # full stack for district 500 in 2021 (county 63.3 + school 194.2 + city 81.4 + arena 0.4)
RATIO_LEGAL_RESIDENCE = 0.04    # S.C. Code 12-43-220(c)
RATIO_OTHER = 0.06              # S.C. Code 12-43-220(e)
HOMESTEAD_CURRENT = 50_000      # S.C. Code 12-37-250 (65+/disabled/blind), tax year 2021
HOMESTEAD_PROPOSED = 75_000     # modeled amount (pending expansion), same eligibility
HOMESTEAD_MODELED = HOMESTEAD_PROPOSED
HOLDOUT_BILL_SHARE = 0.5        # observed bill < 50% of the statutory bill => exempt / FILOT / special arrangement

# Parcel-map export identity columns
PARCEL_ID_COL = 'PIN'
OWNER_NAME_COL = 'OWNAM1'
OWNER_ADDRESS_COL = 'site_address'
PARCEL_URL_TEMPLATE = None      # county property-record portal is session-based; no stable deep link verified

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Fetch / load parcel data

**Source.** Greenville County's own GIS server (`gcgis.org`) could not be reached from the build
environment, so the parcel layer comes from a public ArcGIS Online copy of the county's
`TaxParcel` feature class, published by the Greenville-Pickens Area Transportation Study (GPATS):
`TaxParcel02172021_gdb` (228,190 county parcels). Every record carries `VERDATE` = 2022-09-13 and
payment dates of Dec 2021 – Jan 2022, so the values and `TOTTAX` bills are **tax year 2021**.
The layer has the county CAMA fields needed here: `LANDVAL`, `BLDGVAL`, `TAXMKTVAL` (capped taxable
market value, = `LANDVAL + BLDGVAL`), `FAIRMKTVAL` (uncapped), `TOTTAX` (the full observed bill),
`DIST` (tax district), `RRETCD` (assessment-ratio code), `HMSTCD` (homestead flag), and `LANDUSE`.

**City filter.** User choice: the boundary polygon. The city boundary is the Census TIGERweb
*Incorporated Places* layer (GEOID 4530850, current vintage). A parcel is in the city if its
representative point falls inside the polygon.

| Concept | Column | Notes |
|---|---|---|
| Land value | `LANDVAL` | Taxable (post-cap) market value of land |
| Improvement value | `BLDGVAL` | Taxable (post-cap) market value of buildings |
| Total taxable market value | `TAXMKTVAL` | = `LANDVAL + BLDGVAL`; ≤ `FAIRMKTVAL` because of the 15% reassessment cap |
| Fair market value | `FAIRMKTVAL` | Uncapped appraisal |
| Use / class code | `LANDUSE`, `PROPTYPE` | County land-use code + broad type |
| Assessment ratio | `RRETCD` | `2` = 4% legal residence; `1`/`9`/`6` = 6% |
| Homestead exemption | `HMSTCD` | `*` = 65+/disabled/blind homestead exemption |
| Exempt / FILOT | (derived) | Observed `TOTTAX` far below the statutory bill — see Section 3 |
| Tax district | `DIST` | 500-series = City of Greenville districts |
| Parcel ID | `PIN` | |

In [2]:
# Step 2a: county parcels (cached)
RAW_PATH = DATA_DIR / 'county_parcels_raw.gpq'
if RAW_PATH.exists():
    county = gpd.read_parquet(RAW_PATH)
    print(f"Loaded {len(county):,} county parcels from cache")
else:
    county = get_feature_data_with_geometry(
        dataset_name='TaxParcel02172021_gdb',
        base_url='https://services3.arcgis.com/bjOyhhlaaCIcmO3H/arcgis/rest/services',
        layer_id=0,
        paginate=True,
        verbose=False,
    )
    county.to_parquet(RAW_PATH)
    print(f"Downloaded and cached {len(county):,} county parcels")

# Step 2b: city boundary (Census TIGERweb incorporated place 4530850)
BOUNDARY_PATH = DATA_DIR / 'city_limits_L4.geojson'
if not BOUNDARY_PATH.exists():
    import requests
    _url = ('https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/'
            'Places_CouSub_ConCity_SubMCD/MapServer/4/query')
    _r = requests.get(_url, params={
        'where': "STATE='45' AND PLACE='30850'", 'outFields': 'NAME,GEOID,AREALAND',
        'returnGeometry': 'true', 'outSR': 4326, 'f': 'geojson'}, timeout=60)
    _r.raise_for_status()
    BOUNDARY_PATH.write_text(_r.text)
city_boundary = gpd.read_file(BOUNDARY_PATH).to_crs(4326)
print(city_boundary[['NAME', 'GEOID']].to_string(index=False),
      f"| area {city_boundary.to_crs(2273).area.sum() / 27_878_400:.1f} sq mi")

# Step 2c: keep parcels whose representative point is inside the boundary
_pts = county[['PIN', 'geometry']].copy()
_pts['geometry'] = county.to_crs(3857).representative_point().to_crs(4326)
_inside = gpd.sjoin(_pts, city_boundary[['geometry']], predicate='within').index.unique()
gdf = county.loc[_inside].copy()
gdf = gdf.drop_duplicates(subset='PIN')
print(f"City parcels (boundary filter): {len(gdf):,}")

CITY_DISTRICTS = sorted(d for d in gdf['DIST'].unique() if str(d).startswith('5'))
gdf['in_city_tax_district'] = gdf['DIST'].isin(CITY_DISTRICTS)
print("City tax districts present:", CITY_DISTRICTS)
print(f"Parcels inside the boundary but in a non-city tax district in TY{TAX_YEAR}: "
      f"{(~gdf['in_city_tax_district']).sum():,}")
print(gdf.loc[~gdf['in_city_tax_district'], 'DIST'].value_counts().head(10).to_dict())

# Parcels in city tax districts whose point falls OUTSIDE the boundary (boundary/vintage mismatch)
_city_dist_all = county['DIST'].isin(CITY_DISTRICTS)
print(f"City-district parcels outside the polygon (dropped): {(_city_dist_all & ~county.index.isin(_inside)).sum():,}")

Loaded 228,190 county parcels from cache
           NAME   GEOID
Greenville city 4530850 | area 31.1 sq mi


City parcels (boundary filter): 26,910
City tax districts present: ['500', '501', '502', '511', '512', '515']
Parcels inside the boundary but in a non-city tax district in TY2021: 423
{'200': 118, '235': 106, 'B': 83, '156': 50, '790': 20, '233': 15, '239': 9, '066': 7, '230': 5, '155': 5}
City-district parcels outside the polygon (dropped): 12


In [3]:
# Step 2d: column checks
for _c in ['LANDVAL', 'BLDGVAL', 'TAXMKTVAL', 'FAIRMKTVAL', 'TOTTAX']:
    gdf[_c] = pd.to_numeric(gdf[_c], errors='coerce').fillna(0).clip(lower=0)
gdf['site_address'] = (gdf['STRNUM'].fillna('').astype(str).str.strip() + ' '
                       + gdf['LOCATE'].fillna('').astype(str).str.strip()).str.strip()

print(f"LANDVAL + BLDGVAL == TAXMKTVAL: {np.isclose(gdf['LANDVAL'] + gdf['BLDGVAL'], gdf['TAXMKTVAL'], atol=5).mean():.1%}")
print(f"Capped (TAXMKTVAL < FAIRMKTVAL): {(gdf['TAXMKTVAL'] < gdf['FAIRMKTVAL'] - 5).mean():.1%}")
print(f"$0 land: {(gdf['LANDVAL'] == 0).sum():,}   $0 building: {(gdf['BLDGVAL'] == 0).sum():,}   $0 total: {(gdf['TAXMKTVAL'] == 0).sum():,}")
print(f"Median land ${gdf['LANDVAL'].median():,.0f}   median building ${gdf['BLDGVAL'].median():,.0f}")
print(gdf[['LANDVAL', 'BLDGVAL', 'TAXMKTVAL', 'FAIRMKTVAL', 'TOTTAX']].describe().round(0).T)

LANDVAL + BLDGVAL == TAXMKTVAL: 100.0%
Capped (TAXMKTVAL < FAIRMKTVAL): 31.7%
$0 land: 210   $0 building: 4,465   $0 total: 199
Median land $59,750   median building $131,170
              count      mean        std  min       25%       50%       75%         max
LANDVAL     26910.0  122126.0   363698.0  0.0   25502.0   59750.0  112100.0  19272750.0
BLDGVAL     26910.0  351925.0  1910981.0  0.0   45325.0  131170.0  289432.0  77424850.0
TAXMKTVAL   26910.0  474052.0  2105302.0  0.0   90000.0  207850.0  406618.0  79326350.0
FAIRMKTVAL  26910.0  512174.0  2388456.0  0.0  105982.0  219580.0  419152.0  86970320.0
TOTTAX      26910.0    6150.0    37874.0  0.0     850.0    1989.0    3868.0   1631664.0


## Section 3 — Classify parcels and validate

### 3a. Assessment ratio and homestead

South Carolina assesses locally assessed real property at **4%** of (capped) market value for an
owner-occupied legal residence and **6%** for everything else (rentals, second homes, commercial,
vacant land). The county's `RRETCD` code carries the ratio. The observed bills confirm it: in
district 500, `RRETCD = 2` parcels pay about **8.0 per $1,000** of market value
(4% × ≈200 mills, the lower figure because Act 388 removes school operating millage from legal
residences), while every other code pays about **20.9 per $1,000** (6% × ≈348 mills).

The homestead exemption (`HMSTCD = '*'`, legal residences only) exempts the first
$50,000 of market value in TY2021. It is modeled at **$75,000**. Following the repo's exemption
hierarchy, the exempt amount comes off building value first, then land.

> Note on manufacturing: 10.5% applies to manufacturing and utility property, which the SC
> Department of Revenue assesses. Those values are not in the county parcel layer. Industrial parcels
> here are county-assessed at 6%.

In [4]:
# Step 3a: ratio + homestead flags
gdf['RRETCD'] = gdf['RRETCD'].fillna('').astype(str).str.strip()
gdf['legal_residence'] = (gdf['RRETCD'] == '2').astype(int)
gdf['assessment_ratio'] = np.where(gdf['legal_residence'] == 1, RATIO_LEGAL_RESIDENCE, RATIO_OTHER)
gdf['homestead_flag'] = ((gdf['HMSTCD'].fillna('').astype(str).str.strip() == '*')
                         & (gdf['legal_residence'] == 1)).astype(int)

# Empirical check: implied full-stack mills per $1,000 assessed, by ratio code (district 500, valued parcels)
_d = gdf[(gdf['DIST'] == '500') & (gdf['TAXMKTVAL'] > 30_000) & (gdf['TOTTAX'] > 0)].copy()
_d['implied_mills'] = _d['TOTTAX'] / (_d['TAXMKTVAL'] * _d['assessment_ratio']) * 1000
print(_d.groupby(['RRETCD', 'homestead_flag'])['implied_mills'].describe(percentiles=[.25, .5, .75]).round(1))
print(f"Legal residences: {gdf['legal_residence'].sum():,}   with homestead exemption: {gdf['homestead_flag'].sum():,}")

                        count   mean    std    min    25%    50%    75%     max
RRETCD homestead_flag                                                          
1      0                688.0  306.9  102.1   15.0  342.1  345.9  350.6   740.9
2      0               8828.0  197.8   46.4    2.0  199.1  201.3  205.6   645.0
       1               3077.0  150.7   43.7    4.1  131.3  160.2  178.4   530.2
6      0                  2.0  117.1    0.0  117.1  117.1  117.1  117.1   117.1
9      0               8419.0  288.5  147.6    0.2  177.1  347.5  358.0  4449.1
Legal residences: 13,249   with homestead exemption: 3,408


### 3b. Exempt, FILOT and special-arrangement parcels (held out)

The parcel layer has no explicit exemption or FILOT flag. Exempt parcels (government, churches,
hospitals, universities) **keep their appraised values**, and their `TOTTAX` holds only small
non-ad-valorem charges. FILOT parcels pay a negotiated fee instead of millage. Both are identified
by comparing the **observed bill** with the **statutory bill** the parcel would owe at its tax
district's full-stack millage:

```
statutory_bill = (TAXMKTVAL − homestead_TY2021) × ratio × district_mills / 1000
low_bill       = TOTTAX < 50% × statutory_bill
```

A low bill on its own doesn't settle whether a parcel is exempt, FILOT, or just mis-timed. Step 3c-2
sorts the low-bill parcels once the property categories exist.

`district_mills` is estimated per tax district and ratio class as the median implied rate of
parcels that clearly pay full millage. For district 500 the 6% estimate is close to the published
339.3 mills; the difference comes from stormwater and sanitation charges included in `TOTTAX`.
Held-out parcels are excluded from the revenue-neutral solve, the export and the charts, as the
repo requires, and are reported by count. Parcels inside the boundary that sat in a non-city tax
district in TY2021 are tested against their own district's millage.

In [5]:
# Step 3b: statutory vs observed bill
gdf['hs_exempt_mv_ty2021'] = np.where(gdf['homestead_flag'] == 1,
                                      np.minimum(HOMESTEAD_CURRENT, gdf['TAXMKTVAL']), 0.0)
gdf['assessed_net_ty2021'] = (gdf['TAXMKTVAL'] - gdf['hs_exempt_mv_ty2021']).clip(lower=0) * gdf['assessment_ratio']
gdf['_implied'] = np.where(gdf['assessed_net_ty2021'] > 0,
                           gdf['TOTTAX'] / gdf['assessed_net_ty2021'] * 1000, np.nan)

# Estimate district x class full-stack millage from parcels paying a plausible full rate
_floor = np.where(gdf['legal_residence'] == 1, 100, 200)
_ok = (gdf['_implied'] > _floor) & (gdf['_implied'] < 800) & (gdf['TAXMKTVAL'] > 20_000)
district_mills = gdf[_ok].groupby(['DIST', 'legal_residence'])['_implied'].median().rename('district_mills')
_fallback = gdf[_ok].groupby('legal_residence')['_implied'].median()
gdf = gdf.join(district_mills, on=['DIST', 'legal_residence'])
gdf['district_mills'] = gdf['district_mills'].fillna(gdf['legal_residence'].map(_fallback))
print("Estimated full-stack mills (city districts):")
print(district_mills.loc[[d for d in CITY_DISTRICTS if d in district_mills.index.get_level_values(0)]].round(1).unstack())

gdf['statutory_bill'] = gdf['assessed_net_ty2021'] * gdf['district_mills'] / 1000
gdf['bill_share'] = np.where(gdf['statutory_bill'] > 0, gdf['TOTTAX'] / gdf['statutory_bill'], np.nan)

gdf['low_bill'] = ((gdf['statutory_bill'] >= 100)
                   & (gdf['TOTTAX'] < HOLDOUT_BILL_SHARE * gdf['statutory_bill'])).astype(int)
print(f"\nParcels billed < {HOLDOUT_BILL_SHARE:.0%} of statutory: {gdf['low_bill'].sum():,} "
      "(classified into exempt / FILOT / bill-timing after the categories are built, Step 3c-2)")

Estimated full-stack mills (city districts):
legal_residence      0      1
DIST                         
500              351.5  202.5
501              341.4  197.0
502              343.7  197.7
511              346.1  198.8
512              354.2  204.2
515              342.8  200.0

Parcels billed < 50% of statutory: 3,880 (classified into exempt / FILOT / bill-timing after the categories are built, Step 3c-2)


### 3c. Property categories

The categories come from the county `LANDUSE` code. Its meaning was checked by reading owner and
site names for each code (for example, 240/250/270 are hotels, 371 and 821 are surface parking,
and 810 is churches), with `PROPTYPE` as the fallback. A residential `1100` parcel counts as a
**condominium** when its subdivision or description names a condo or unit. Greenville condos carry
real land value (median land share about 16%), so no St. Paul–style condo collapse is needed.
Any parcel with $0 building value becomes **Vacant Land**, except the parking codes, where a
paved lot with no building is the expected record.

In [6]:
# Step 3c: categories
LANDUSE_MAP = {
    '1100': 'Single Family Residential',
    '1101': 'Other Residential',
    '1180': 'Vacant Land',
    '1181': 'Other',                         # residential common area
    '110': 'Small Multi-Family (2-4 units)',
    '112': 'Small Multi-Family (2-4 units)',
    '120': 'Large Multi-Family (5+ units)',
    '122': 'Large Multi-Family (5+ units)',
    '141': 'Other Commercial',               # assisted living / memory care
    '205': 'Other',                          # commercial common area
    '240': 'Hotel', '250': 'Hotel', '270': 'Hotel',
    '370': 'Transportation - Parking',       # parking decks
    '371': 'Transportation - Parking',       # surface lots
    '821': 'Transportation - Parking',       # surface lots / airport parking
    '6800': 'Vacant Land',                   # vacant commercial
    '770': 'Other Commercial',               # recreation / arena
    '810': 'Other',                          # churches
    '850': 'Other',                          # schools / institutional
    '851': 'Other Commercial',
    '872': 'Other Commercial',               # funeral homes
    '910': 'Other Commercial',               # self-storage
}
OFFICE_CODES = {'409', '410', '411', '420', '421', '423', '425', '430', '431', '720'}
RETAIL_PREFIXES = ('3', '5', '6')            # 300-360 auto/gas/car wash, 510-590 retail, 610-632 restaurants


def classify(row) -> str:
    code = str(row['LANDUSE']).strip()
    if code == '1100':
        desc = f"{row.get('SUBDIV') or ''} {row.get('DESCR') or ''}".upper()
        if any(k in desc for k in ('CONDO', 'HPR', 'UNIT', 'HORIZONTAL')):
            return 'Condominium'
        return 'Single Family Residential'
    if code in LANDUSE_MAP:
        return LANDUSE_MAP[code]
    if code in OFFICE_CODES:
        return 'Office / Commercial Condo'
    if len(code) == 3 and code.startswith(RETAIL_PREFIXES) and code not in ('370', '371'):
        return 'Retail / General Commercial'
    if len(code) == 3 and code.startswith('9') and code >= '940':
        return 'Industrial'
    pt = str(row.get('PROPTYPE') or '').upper()
    return {
        'RESIDENTIAL': 'Other Residential', 'MULTI-FAMILY': 'Large Multi-Family (5+ units)',
        'COMMERCIAL': 'Other Commercial', 'INDUSTRIAL': 'Industrial',
        'AGRICULTURAL': 'Agricultural', 'MOBILE HOME': 'Other Residential',
    }.get(pt, 'Other')


gdf['PROPERTY_CATEGORY'] = gdf.apply(classify, axis=1)
_parking = gdf['PROPERTY_CATEGORY'] == 'Transportation - Parking'
gdf.loc[(gdf['BLDGVAL'] <= 0) & ~_parking, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print(gdf['PROPERTY_CATEGORY'].value_counts())

PROPERTY_CATEGORY
Single Family Residential         17950
Vacant Land                        4376
Condominium                        1276
Office / Commercial Condo          1240
Retail / General Commercial         937
Small Multi-Family (2-4 units)      457
Industrial                          193
Transportation - Parking            157
Large Multi-Family (5+ units)       139
Other Commercial                     80
Other                                58
Hotel                                33
Other Residential                    14
Name: count, dtype: int64


### 3c-2. Resolve the low-bill parcels: exempt, FILOT, or bill timing

The parcel snapshot is dated September 2022, but `TOTTAX` is the **TY2021** bill. A parcel that
was sold, completed, newly platted, or changed legal-residence status during 2021–22 can carry
current attributes alongside a bill computed on its old status. A low bill therefore means
different things for different parcels:

| Rule | Treatment |
|---|---|
| Owner is government, church, school, university, hospital, housing authority or a known nonprofit, **and** billed < 50% of statutory | **Exempt** — held out |
| Non-residential category (office, retail, hotel, industrial, 5+ apartments, parking, other commercial, commercial vacant land), billed < 50% of statutory, owner not exempt | **FILOT / special arrangement** — held out (user decision) |
| Residential category (single family, condo, 2–4 units, residential lots), owner not exempt | **Kept** and modeled at its statutory value. The low bill reflects timing (new construction, sale, new plat), not a tax arrangement. FILOTs are not used for owner-occupied housing |
| `TAXMKTVAL = 0` | **No value** — held out |

In [7]:
# Step 3c-2: exempt / FILOT / timing
EXEMPT_OWNER_PATTERN = (
    r"CITY OF|\bCOUNTY\b|GREENVILLE CO\b|GVILLE CO\b|STATE OF|SOUTH CAROLINA DEPT|\bS ?C DEPT|UNITED STATES|\bUSA\b|"
    r"HOUSING AUTH|AUTHORITY|COMMISSION|SCHOOL|UNIVERSITY|COLLEGE|\bTECH\b|HOSPITAL|HEALTH SYS|BON SECOURS|PRISMA|"
    r"CHURCH|\bCH\b|BAPTIST|METHODIST|PRESBYTERIAN|CATHOLIC|EPISCOPAL|LUTHERAN|DIOCESE|PARISH|CONGREGATION|"
    r"MINISTR|TEMPLE|MOSQUE|SYNAGOGUE|WORSHIP|TABERNACLE|FELLOWSHIP|CEMETERY|"
    r"HABITAT|HOMES OF HOPE|GENESIS HOMES|UNITED HOUSING|HOUSING FUND|FOUNDATION|Y ?M ?C ?A|MUSEUM|ARENA DISTRICT|"
    r"COMMUNITY ECONOMIC|COMMUNITY DEV|\bINC\b.*\bTRS\b|TRUSTEES"
)
RESIDENTIAL_CATS = {'Single Family Residential', 'Condominium', 'Small Multi-Family (2-4 units)', 'Other Residential'}
_owner = gdf['OWNAM1'].fillna('').astype(str).str.upper()
gdf['exempt_owner'] = _owner.str.contains(EXEMPT_OWNER_PATTERN, regex=True)
_is_resid = gdf['PROPERTY_CATEGORY'].isin(RESIDENTIAL_CATS) | (
    (gdf['PROPERTY_CATEGORY'] == 'Vacant Land') & gdf['LANDUSE'].astype(str).str.startswith('11'))

gdf['no_value'] = (gdf['TAXMKTVAL'] <= 0).astype(int)
gdf['exempt'] = ((gdf['low_bill'] == 1) & gdf['exempt_owner']).astype(int)
gdf['filot_or_special'] = ((gdf['low_bill'] == 1) & ~gdf['exempt_owner'] & ~_is_resid).astype(int)
gdf['bill_timing_mismatch'] = ((gdf['low_bill'] == 1) & ~gdf['exempt_owner'] & _is_resid).astype(int)
gdf['full_exmp'] = ((gdf['no_value'] == 1) | (gdf['exempt'] == 1) | (gdf['filot_or_special'] == 1)).astype(int)

_tot_mv = gdf['TAXMKTVAL'].sum()
for _flag, _label in [('no_value', 'No taxable value (new plats / common elements)'),
                      ('exempt', 'Exempt owner'),
                      ('filot_or_special', 'FILOT / special arrangement (non-residential)'),
                      ('bill_timing_mismatch', 'Residential low bill — KEPT (timing)')]:
    _m = gdf[_flag] == 1
    print(f"{_label:<50} {_m.sum():>6,} parcels  ${gdf.loc[_m, 'TAXMKTVAL'].sum() / 1e9:5.2f}B  ({gdf.loc[_m, 'TAXMKTVAL'].sum() / _tot_mv:.1%} of value)")

print("\nHeld out by category:")
print(gdf[gdf['full_exmp'] == 1].groupby('PROPERTY_CATEGORY').agg(
    n=('PIN', 'size'), market_value=('TAXMKTVAL', 'sum')).sort_values('market_value', ascending=False))
print("\nLargest FILOT / special-arrangement parcels:")
print(gdf[gdf['filot_or_special'] == 1].nlargest(12, 'TAXMKTVAL')[
    ['PIN', 'OWNAM1', 'PROPERTY_CATEGORY', 'TAXMKTVAL', 'TOTTAX', 'statutory_bill', 'bill_share']].round(2).to_string(index=False))
print("\nBill share of parcels kept (should cluster near 1.0):")
print(gdf.loc[gdf['full_exmp'] == 0, 'bill_share'].describe(percentiles=[.05, .25, .5, .75, .95]).round(3))

print("\nModeled parcels by category:")
print(gdf.loc[gdf['full_exmp'] == 0, 'PROPERTY_CATEGORY'].value_counts())
_resid = gdf[(gdf['full_exmp'] == 0) & gdf['PROPERTY_CATEGORY'].isin(['Other', 'Other Commercial', 'Other Residential'])]
print("\nCodes inside the residual buckets (taxable parcels):")
print(_resid.groupby(['PROPERTY_CATEGORY', 'LANDUSE']).size().sort_values(ascending=False).head(10))

No taxable value (new plats / common elements)        199 parcels  $ 0.00B  (0.0% of value)
Exempt owner                                        1,002 parcels  $ 0.51B  (4.0% of value)
FILOT / special arrangement (non-residential)         366 parcels  $ 0.90B  (7.0% of value)
Residential low bill — KEPT (timing)                2,512 parcels  $ 0.81B  (6.3% of value)

Held out by category:
                                  n  market_value
PROPERTY_CATEGORY                                
Office / Commercial Condo       120     546366710
Vacant Land                     956     301384010
Large Multi-Family (5+ units)    23     231641010
Transportation - Parking         59      89876560
Retail / General Commercial      57      84157780
Industrial                       39      38600560
Other                            33      31859120
Other Commercial                 12      31683380
Single Family Residential       232      31414770
Hotel                             2      12852495
Small Mul

### 3d. Validate against the observed bills (parcel-level ground truth)

The layer carries each parcel's actual TY2021 bill, so the reconstruction is checked directly. For
taxable 6%-class parcels in district 500, the published full-stack rate (339.3 mills) times the
assessed value should reproduce the observed `TOTTAX` to within a few percent. The remainder is
stormwater and sanitation charges, partial-year changes, and rollback taxes.

In [8]:
# Step 3d: published-millage reconstruction of observed bills, district 500, 6% class, taxable
# (parcels whose TY2021 bill predates their current attributes — Step 3c-2 — are excluded from this check)
_v = gdf[(gdf['DIST'] == '500') & (gdf['legal_residence'] == 0) & (gdf['full_exmp'] == 0)
         & (gdf['bill_timing_mismatch'] == 0)]
_recon = (_v['TAXMKTVAL'] * RATIO_OTHER * DISTRICT_500_TOTAL_MILLAGE / 1000).sum()
_obs = _v['TOTTAX'].sum()
BILL_GAP_PCT = (_recon / _obs - 1) * 100
print(f"District 500, 6% class: reconstructed ${_recon:,.0f} vs observed ${_obs:,.0f}  gap {BILL_GAP_PCT:+.2f}%")

# Legal residences: implied non-school-operating stack
_lr = gdf[(gdf['DIST'] == '500') & (gdf['legal_residence'] == 1) & (gdf['homestead_flag'] == 0)
          & (gdf['full_exmp'] == 0) & (gdf['bill_timing_mismatch'] == 0)]
_lr_mills = _lr['TOTTAX'].sum() / (_lr['TAXMKTVAL'] * RATIO_LEGAL_RESIDENCE).sum() * 1000
print(f"District 500 legal residences: implied {_lr_mills:.1f} mills (= 339.3 less school operating, "
      f"implying ~{DISTRICT_500_TOTAL_MILLAGE - _lr_mills:.0f} school-operating mills removed by Act 388)")

District 500, 6% class: reconstructed $86,645,117 vs observed $87,822,944  gap -1.34%
District 500 legal residences: implied 198.0 mills (= 339.3 less school operating, implying ~141 school-operating mills removed by Act 388)


## Section 4 — Current city tax

The city levy is rebuilt for every taxable parcel inside the boundary:

```
assessed_land        = LANDVAL × ratio
assessed_improvement = BLDGVAL × ratio
homestead (assessed) = min(HOMESTEAD, TAXMKTVAL) × ratio     (improvements first, then land)
current city tax     = (assessed_land + assessed_improvement − homestead) × 81.4 / 1000
```

Two versions are computed: `current_tax_hs50k` (TY2021 law) and `current_tax` (the $75K modeled
baseline). In South Carolina the state reimburses local governments for the homestead exemption
through the Homestead Exemption Fund. Raising it to $75K therefore lowers what homestead owners
pay **without** raising anyone else's city rate. The 81.4-mill rate is held fixed, and the LVT is
revenue-neutral against the taxpayer-paid city levy at $75K.

**Revenue control.** The City's FY2022 Popular Annual Financial Report shows total city **tax**
collections of $54.7M. That figure also covers vehicle and business personal property, FILOT fees
and TIF increment, so the modeled **real-property** levy should come in below it. The parcel-level
check in 3d is the primary validation. No published city real-property-only levy figure was
reachable from the build environment.

In [9]:
# Step 4: current city tax
gdf['assessed_land'] = gdf['LANDVAL'] * gdf['assessment_ratio']
gdf['assessed_improvement'] = gdf['BLDGVAL'] * gdf['assessment_ratio']
gdf['millage_rate'] = CITY_MILLAGE


def homestead_assessed(amount: float) -> pd.Series:
    mv = np.where(gdf['homestead_flag'] == 1, np.minimum(amount, gdf['TAXMKTVAL']), 0.0)
    return pd.Series(mv, index=gdf.index) * gdf['assessment_ratio']


gdf['homestead_assessed_50k'] = homestead_assessed(HOMESTEAD_CURRENT)
gdf['homestead_assessed_75k'] = homestead_assessed(HOMESTEAD_PROPOSED)
gdf['homestead_assessed'] = homestead_assessed(HOMESTEAD_MODELED)

rev_50k, _, _tmp = calculate_current_tax(
    df=gdf, tax_value_col='TAXMKTVAL', millage_rate_col='millage_rate',
    land_value_col='assessed_land', improvement_value_col='assessed_improvement',
    exemption_col='homestead_assessed_50k', exemption_flag_col='full_exmp',
)
gdf['current_tax_hs50k'] = _tmp['current_tax']

current_revenue, _, gdf = calculate_current_tax(
    df=gdf, tax_value_col='TAXMKTVAL', millage_rate_col='millage_rate',
    land_value_col='assessed_land', improvement_value_col='assessed_improvement',
    exemption_col='homestead_assessed', exemption_flag_col='full_exmp',
)

PAFR_FY2022_TAXES = 54_700_000
print(f"Modeled city real-property levy, TY{TAX_YEAR} law ($50K homestead): ${rev_50k:,.0f}")
print(f"Modeled city real-property levy, baseline ($75K homestead):      ${current_revenue:,.0f}")
print(f"Homestead increase cost (state-reimbursed):                       ${rev_50k - current_revenue:,.0f}")
print(f"Taxable assessed value (pre-homestead): ${(gdf.loc[gdf['full_exmp'] == 0, 'assessed_land'] + gdf.loc[gdf['full_exmp'] == 0, 'assessed_improvement']).sum():,.0f}")
print(f"Share of FY2022 PAFR total city tax collections (${PAFR_FY2022_TAXES / 1e6:.1f}M, all property types): "
      f"{rev_50k / PAFR_FY2022_TAXES:.1%}")
_in = gdf['in_city_tax_district'] & (gdf['full_exmp'] == 0)
print(f"  of which parcels in TY{TAX_YEAR} city tax districts: ${gdf.loc[_in, 'current_tax_hs50k'].sum():,.0f}; "
      f"parcels annexed since / outside city districts: ${gdf.loc[~gdf['in_city_tax_district'] & (gdf['full_exmp'] == 0), 'current_tax_hs50k'].sum():,.0f}")

Modeled city real-property levy, TY2021 law ($50K homestead): $47,351,829
Modeled city real-property levy, baseline ($75K homestead):      $47,090,182
Homestead increase cost (state-reimbursed):                       $261,647
Taxable assessed value (pre-homestead): $588,496,184
Share of FY2022 PAFR total city tax collections ($54.7M, all property types): 86.6%
  of which parcels in TY2021 city tax districts: $46,083,956; parcels annexed since / outside city districts: $1,267,873


## Section 5 — 4:1 split-rate model

The split-rate is solved on **assessed** land and assessed improvements (ratio already applied),
net of the $75K homestead exemption (improvements first). This keeps the 4% vs 6% class
difference intact, so the only change is land vs buildings. Fully exempt, FILOT and no-value
parcels are held out and excluded from the export.

In [10]:
# Step 5: split-rate
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='assessed_land',
    improvement_value_col='assessed_improvement',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
    exemption_col='homestead_assessed',
)

print(f"Held out {n_exempt:,} exempt / FILOT / no-value parcels (excluded from model, export, and charts).")
print(f"Modeled parcels: {len(gdf):,}")
print(f"Current city millage: {CITY_MILLAGE:.2f}")
print(f"Land millage: {land_millage:.4f}   Improvement millage: {improvement_millage:.4f}")
print(f"Revenue check: ${new_revenue:,.0f} vs current ${gdf['current_tax'].sum():,.0f}")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate Tax Impact (city levy)")

Held out 1,567 exempt / FILOT / no-value parcels (excluded from model, export, and charts).
Modeled parcels: 25,343
Current city millage: 81.40
Land millage: 186.5334   Improvement millage: 46.6334
Revenue check: $47,090,182 vs current $47,090,182

greenville_sc — 4:1 Split-Rate Tax Impact (city levy)
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
     Single Family Residential  17718        $568,571        3.0%        $32          $18   13.3%       3.4%            41.2%            26.7%
                   Vacant Land   3420      $2,161,184      129.2%       $632         $129  129.1%     129.2%           100.0%             0.0%
                   Condominium   1276       $-115,291       -7.5%       $-90         $-82   -8.3%     -13.7%            11.5%            64.3%
     Office / Commercial Condo   1120       $-743,977      -11.2%      $-664          $84   10.7%       5.8%            44.3%

In [11]:
# Step 5b: artifact scan (validate.md Gate 5)
_d = gdf.copy()
_d['bldg_share'] = _d['taxable_improvement_value'] / (
    _d['taxable_land_value'] + _d['taxable_improvement_value']).replace(0, np.nan)
scan = _d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('BLDGVAL', lambda s: (s <= 1000).mean()),
    current_tax=('current_tax', 'sum'),
    new_tax=('new_tax', 'sum'),
).sort_values('median_pct', ascending=False)
_ceiling = land_millage / CITY_MILLAGE - 1
print(f"Land-only ceiling: {_ceiling * 100:+.1f}%   building-only floor: {(improvement_millage / CITY_MILLAGE - 1) * 100:+.1f}%")
print(scan.round(3).to_string())

Land-only ceiling: +129.2%   building-only floor: -42.7%
                                    n  median_pct  median_bldg_share  pct_zero_bldg   current_tax       new_tax
PROPERTY_CATEGORY                                                                                              
Vacant Land                      3420     129.157              0.000          1.000  1.673306e+06  3.834490e+06
Transportation - Parking           98     124.694              0.026          0.459  2.081249e+05  3.409407e+05
Other Residential                  12      81.224              0.279          0.167  1.595961e+03  2.722465e+03
Other                              25      43.412              0.499          0.280  4.057976e+04  5.654182e+04
Retail / General Commercial       880      23.950              0.612          0.000  6.443759e+06  7.144920e+06
Industrial                        154      16.074              0.658          0.000  6.352620e+05  6.663372e+05
Small Multi-Family (2-4 units)    425      11.7

## Section 6 — Homestead: $50K today vs. $75K, and exploration charts

The **$50K→$75K** comparison under today's system (81.4 mills, no LVT) isolates the homestead
change from the LVT. The main LVT comparison above uses $75K in both the baseline and the reform.

In [12]:
# Step 6a: homestead change under the current system
hs = gdf[gdf['homestead_flag'] == 1].copy()
hs['hs_saving'] = hs['current_tax_hs50k'] - hs['current_tax']
print(f"Homestead parcels modeled: {len(hs):,}")
print(f"City tax on homestead parcels: ${hs['current_tax_hs50k'].sum():,.0f} at $50K -> ${hs['current_tax'].sum():,.0f} at $75K "
      f"(saving ${hs['hs_saving'].sum():,.0f}, reimbursed by the state)")
print(f"Median city-tax saving per homestead parcel: ${hs['hs_saving'].median():,.0f}  "
      f"(max possible = $25,000 x 4% x 81.4/1000 = ${25_000 * 0.04 * CITY_MILLAGE / 1000:,.0f})")
print(f"Homestead parcels paying $0 city tax: {(hs['current_tax_hs50k'] <= 0.5).sum():,} at $50K -> {(hs['current_tax'] <= 0.5).sum():,} at $75K")
print(f"Homestead parcels under the 4:1 LVT (vs $75K baseline): median change {hs.loc[hs['current_tax'] > 0, 'tax_change_pct'].median():+.1f}%, "
      f"total ${hs['current_tax'].sum():,.0f} -> ${hs['new_tax'].sum():,.0f}")
homestead_comparison = pd.DataFrame({
    'Scenario': ['Today: $50K homestead, 81.4 mills', 'Baseline: $75K homestead, 81.4 mills', '4:1 LVT with $75K homestead'],
    'City tax on homestead parcels': [hs['current_tax_hs50k'].sum(), hs['current_tax'].sum(), hs['new_tax'].sum()],
    'Median per parcel': [hs['current_tax_hs50k'].median(), hs['current_tax'].median(), hs['new_tax'].median()],
}).round(0)
print(homestead_comparison.to_string(index=False))

Homestead parcels modeled: 3,407
City tax on homestead parcels: $2,710,590 at $50K -> $2,448,943 at $75K (saving $261,647, reimbursed by the state)
Median city-tax saving per homestead parcel: $81  (max possible = $25,000 x 4% x 81.4/1000 = $81)
Homestead parcels paying $0 city tax: 68 at $50K -> 361 at $75K
Homestead parcels under the 4:1 LVT (vs $75K baseline): median change +31.7%, total $2,448,943 -> $2,865,869
                            Scenario  City tax on homestead parcels  Median per parcel
   Today: $50K homestead, 81.4 mills                      2710590.0              549.0
Baseline: $75K homestead, 81.4 mills                      2448943.0              467.0
         4:1 LVT with $75K homestead                      2865869.0              607.0


**Sensitivity: how the homestead exemption is split between land and building.** The repo's
standard hierarchy takes dollar exemptions off **buildings first**. Under a split-rate that matters:
the $75K uses up the low-taxed building value, so the remaining value is mostly land at the 4×
rate. SC law exempts "the first $X of fair market value" without allocating it between land and
building. The run below allocates the exemption **pro rata** to the land/building mix and re-solves.
It is a sensitivity only. The exported result uses the standard improvements-first hierarchy.

In [13]:
# Step 6a-2: pro-rata homestead allocation sensitivity
_s = gdf.copy()
_tot = (_s['assessed_land'] + _s['assessed_improvement']).replace(0, np.nan)
_s['pr_land'] = (_s['assessed_land'] - _s['homestead_assessed'] * _s['assessed_land'] / _tot).fillna(0).clip(lower=0)
_s['pr_imp'] = (_s['assessed_improvement'] - _s['homestead_assessed'] * _s['assessed_improvement'] / _tot).fillna(0).clip(lower=0)
_lm, _im, _, _s = model_split_rate_tax(
    df=_s, land_value_col='pr_land', improvement_value_col='pr_imp',
    current_revenue=_s['current_tax'].sum(), land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)
_hs = _s['homestead_flag'] == 1
homestead_sensitivity = pd.DataFrame({
    'Allocation': ['Improvements first (exported)', 'Pro rata (sensitivity)'],
    'Land millage': [land_millage, _lm],
    'Improvement millage': [improvement_millage, _im],
    'Homestead median % change': [gdf.loc[(gdf['homestead_flag'] == 1) & (gdf['current_tax'] > 0), 'tax_change_pct'].median(),
                                  _s.loc[_hs & (_s['current_tax'] > 0), 'tax_change_pct'].median()],
    'Homestead total city tax': [gdf.loc[gdf['homestead_flag'] == 1, 'new_tax'].sum(), _s.loc[_hs, 'new_tax'].sum()],
    'SFR median % change': [gdf.loc[gdf['PROPERTY_CATEGORY'] == 'Single Family Residential', 'tax_change_pct'].median(),
                            _s.loc[_s['PROPERTY_CATEGORY'] == 'Single Family Residential', 'tax_change_pct'].median()],
}).round(2)
print(homestead_sensitivity.to_string(index=False))

                   Allocation  Land millage  Improvement millage  Homestead median % change  Homestead total city tax  SFR median % change
Improvements first (exported)        186.53                46.63                      31.66                2865869.14                 3.40
       Pro rata (sensitivity)        187.86                46.96                       2.98                2552002.70                 1.19


In [14]:
# Step 6b: quick category preview (not part of the standard report)
fig, ax = plt.subplots(figsize=(10, 6))
_m = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
_m.plot.barh(ax=ax, color=['#2E8B57' if v < 0 else '#B22222' for v in _m])
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Greenville, SC — Median City-Tax Change by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()

## Section 7 — Census join, standard export, report, parcel map

In [15]:
# Census join — OPTIONAL, must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
_census_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct']
if not os.getenv('CENSUS_API_KEY'):
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in _census_cols:
        gdf[_col] = float('nan')
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=90)
                gdf = match_to_census_blockgroups(gdf, census_gdf)
                # census_gdf already carries demographics — no second merge (avoids _x/_y duplicates)
                if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                    gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
                if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                    gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
                print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
            except concurrent.futures.TimeoutError:
                print("Census API timed out — skipping census join")
                for _col in _census_cols:
                    gdf[_col] = float('nan')
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in _census_cols:
            gdf[_col] = float('nan')

Census join: 100.0% matched


In [16]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export — GeoParquet + self-contained interactive HTML map
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

  ✓ greenville_sc: 25,343 rows → ../../analysis/data/greenville_sc.csv  [model: split_rate_4to1]
create_city_report: excluded 251 fully-exempt parcels (25,343 → 25,092 modeled).


  ✓ greenville_sc: 25,343 parcels → ../../analysis/maps/greenville_sc.parquet  [map export; 0 with record links]


  ✓ greenville_sc: interactive map → ../../analysis/reports/greenville_sc/parcel_map.html  [25,343 parcels]
Done.


## Validation summary and limitations

| Check | Result |
|---|---|
| Revenue match (Gate 1) | Published TY2021 district-500 millage (339.3) × modeled 6% assessed value reproduces observed bills at **−1.3%** (remainder = stormwater/sanitation fees). Legal residences imply ~198 mills (= 339.3 less Act 388 school operating). Modeled city real-property levy **$47.4M** at $50K homestead (**$47.1M** at $75K) = 87% of FY2022 total city tax collections ($54.7M, PAFR, which also includes vehicles, business personal property, FILOT fees and TIF) |
| Parcel count | 26,910 parcels inside the boundary → **25,343 modeled**. Held out: 1,002 exempt-owner, 366 FILOT/special (non-residential), 199 no-value |
| Distribution (Gate 2) | Vacant land +129% (= land-only ceiling), parking +125%, retail +24%, SFR **+3.4%** median (+3.8% in the report chart, which drops the 251 parcels with $0 current tax), condos −14%, hotels −20%, 5+ apartments −25% |
| Census coverage (Gate 3) | 100% of parcels joined to 2022 block groups (85% have non-null median income) |
| PNGs (Gate 4) | 7 of 7, plus `parcel_map.html` and `analysis/maps/greenville_sc.parquet` (no parcel links: `PARCEL_URL_TEMPLATE = None`) |
| Artifact scan (Gate 5) | Building shares plausible by category (retail 61%, office 72%, apartments 90%, SFR 73%). Only vacant land and parking (which really have ~0% building) sit at the ceiling. First pass flagged 2,041 SFRs as "exempt"; traced to TY2021 bills that predate the Sept-2022 attributes. Fixed with the owner/category rule in 3c-2 |

**Why single-family homes rise slightly.** Greenville's taxable base is 25% land overall, and
single-family homes are 26% land. Under a 4:1 split, the categories that gain are the
building-heavy ones: downtown apartments, hotels and condos, which carry 83–90% building share.
Homes sit almost exactly at the city average, so their median moves little. The large
transfers run from vacant lots and surface parking toward apartments and hotels.

**Homestead interaction.** Homestead (65+/disabled/blind) parcels show a median **+32%** because
the repo's standard hierarchy takes the $75K off building value first. The remaining value is
then mostly land at the 4× rate. Allocating the exemption pro rata to the land/building mix
(Section 6 sensitivity) gives **+3%**. Any real legislation would have to make this choice
explicitly.

**Limitations**

- **Data vintage.** TY2021 values and bills (county snapshot of Sept 2022, ArcGIS Online copy).
  Greenville County has since reassessed, and the city millage was still 81.4 in 2025–26 per
  the county auditor. The land/building mix, not the dollar level, drives the split-rate results,
  but absolute dollar figures are 2021 dollars.
- **Exempt/FILOT detection is inferred.** The layer has no exemption or FILOT flag. Held-out
  parcels were found by comparing each bill with its statutory amount, plus owner names. A few
  new commercial buildings billed on land only in 2021 may be wrongly held out as "FILOT", and a
  few exempt nonprofits with unrecognized names may be kept.
- **Bill-timing residentials (2,512 parcels)** are modeled at current attributes, not at their
  stale 2021 bill.
- **Boundary vs. tax district.** 423 parcels inside today's TIGER boundary were in non-city tax
  districts in TY2021 (later annexations or boundary differences). They are modeled as city
  parcels at 81.4 mills ($1.3M of the levy). 12 city-district parcels fall just outside the polygon.
- **Personal property, vehicles, and DOR-assessed manufacturing/utility property** (10.5%) are not
  in the parcel layer and are not shifted.
- **Legality.** South Carolina's constitution (Art. X, §1) fixes the assessment ratios and requires
  uniform millage within a class. A city split-rate would need state constitutional or statutory
  change; this is an impact model, not a legal pathway.
- **$75K homestead** is modeled with today's eligibility. Under the 2026 budget proviso (temporary,
  TY2026) the extra $25K went to owners already eligible in 2025. S.768, which passed the Senate
  and is pending in House Ways & Means, has different amounts and residency tiers. In both cases
  the state reimburses localities, so the city millage is held fixed.